# CDLD tutorial — applying it to your own data

This notebook is **not** about reproducing the paper. It is a minimal template
for moving cyclic dual latent discovery (CDLD) onto your own data. It uses
synthetic data and finishes in a few minutes on CPU.

**When to use it.** Two kinds of entities interact, you want to predict the
outcome of the interaction, and **one of the entities has no observed
attributes**. That entity is represented from interaction records alone.

| Domain | Sparse-record entity A | Rich-record entity B | Target |
|---|---|---|---|
| Length of stay | patient | physician | days in hospital |
| Educational assessment | item | student | correct / incorrect |
| Recommendation | product | customer | rating, purchase |
| Work management | task, team | employee | completion time, quality |

**What to change.** Only the two cells marked *여기만 바꾸면 됩니다*
(*change only here*). Everything else runs as is.

**Reproducing the paper** is done by `cdld_los.ipynb`, which requires
MIMIC-IV credentialing.

---

# CDLD 튜토리얼 — 자기 데이터에 적용하기

이 노트북은 논문 결과를 재현하는 것이 목적이 **아닙니다.** 순환 이중 잠재특성
발견(cyclic dual latent discovery, CDLD)을 자기 데이터에 옮기기 위한 최소 템플릿입니다.
가상 데이터를 쓰므로 CPU에서 수 분 안에 끝납니다.

## 언제 쓰나

두 종류의 개체가 상호작용하고 그 결과를 예측하려는데, **한쪽 개체에 관측 특성이
없을 때** 씁니다. 그 개체를 상호작용 기록만으로 표현합니다.

| 도메인 | 기록 희소 개체 A | 기록 풍부 개체 B | 예측 대상 |
|---|---|---|---|
| 재원일수 예측 | 환자 | 의사 | 재원일수 |
| 교육 평가 | 문항 | 학생 | 정오답 |
| 추천 | 상품 | 고객 | 평점·구매 |
| 업무 관리 | 과업·팀 | 직원 | 완료 시간·품질 |

## 바꿔야 할 곳

**「여기만 바꾸면 됩니다」로 표시된 두 개 셀**만 자기 데이터에 맞추면 됩니다.
나머지는 그대로 두어도 동작합니다.

## 논문 재현은?

논문 수치는 `cdld_los.ipynb`가 만듭니다. MIMIC-IV 자격이 필요합니다.

## Setup

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.layers import (Input, Dense, concatenate, Dropout,
                                     BatchNormalization)
from tensorflow.keras.models import Model
from tensorflow.keras.regularizers import l2
from tensorflow.keras.optimizers import Adam
from keras import layers
from sklearn.metrics import mean_squared_error, r2_score

SEED = 0
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
print("TensorFlow", tf.__version__)

## 1. Data / 데이터

In [ ]:
# ================== 여기만 바꾸면 됩니다 (1/2) — 데이터 ==================
# 필요한 것은 상호작용 표 하나뿐입니다.
#   entity_a : 기록이 **적은** 쪽의 정수 인덱스 (예: 환자)
#   entity_b : 기록이 **많은** 쪽의 정수 인덱스 (예: 의사)
#   features : entity_a 의 관측 특성 행렬 (없으면 None)
#   target   : 예측할 연속값
#
# 아래는 설명용 가상 데이터입니다. 자기 데이터로 바꾸면 그대로 돌아갑니다.
N_A, N_B, N_ROWS = 3000, 200, 30000     # A는 기록 희소, B는 기록 풍부
D_FEAT = 12

rng = np.random.default_rng(SEED)
entity_a = rng.integers(0, N_A, N_ROWS)
entity_b = rng.integers(0, N_B, N_ROWS)
features = rng.normal(size=(N_ROWS, D_FEAT)).astype(np.float32)

# 참값 구조: 관측 특성의 선형 효과 + B의 숨은 성향 + A×B 상호작용 + 잡음
w = rng.normal(size=D_FEAT)
b_effect = rng.normal(scale=0.8, size=N_B)          # B의 숨은 성향 (관측 불가)
a_style = rng.normal(scale=0.5, size=N_A)
target = (features @ w * 0.5 + b_effect[entity_b]
          + 0.3 * a_style[entity_a] * b_effect[entity_b]
          + rng.normal(scale=1.0, size=N_ROWS)).astype(np.float32)

print(f"상호작용 {N_ROWS:,}건 | A {N_A:,}개(평균 {N_ROWS/N_A:.1f}건) "
      f"| B {N_B:,}개(평균 {N_ROWS/N_B:.1f}건)")
print("A는 기록이 희소하고 B는 풍부합니다 — 이 비대칭이 설계를 결정합니다.")

In [ ]:
# 학습/검증/평가 분할
idx = rng.permutation(N_ROWS)
n1, n2 = int(N_ROWS * 0.7), int(N_ROWS * 0.8)
tr, va, te = idx[:n1], idx[n1:n2], idx[n2:]
print(f"train {len(tr):,} / valid {len(va):,} / test {len(te):,}")

# 주의: 개체 단위 잠재특성은 그 개체가 학습 분할에 등장해야 정의됩니다
# (transductive). 학습에 없던 개체는 예측 시 평균 벡터 등으로 대체해야 합니다.
unseen_b = len(set(entity_b[te]) - set(entity_b[tr]))
print(f"평가 분할에만 있는 B: {unseen_b}개")

## 2. CDLD components / CDLD 구성요소

In [ ]:
# ================== CDLD 구성요소 (그대로 두면 됩니다) ==================
class ValueOfIndexLayer(layers.Layer):
    def __init__(self, index, **kw):
        super().__init__(**kw); self.index = index
    def call(self, x):
        return tf.cast(x[:, self.index], tf.int32)

class IndexedValueLayer(layers.Layer):
    """개체 하나당 한 행인 잠재특성 테이블. trainable 플래그로 한쪽을 얼린다."""
    def __init__(self, value_shape, trainable=True, **kw):
        super().__init__(**kw)
        self.value_shape = value_shape; self._trainable_flag = trainable
    def build(self, _):
        self.values = self.add_weight(shape=self.value_shape,
                                      initializer="random_uniform",
                                      trainable=self._trainable_flag, name="values")
    def call(self, idx): return tf.gather(self.values, idx)
    def set_value(self, v): self.values.assign(v)
    def get_value(self): return self.values.numpy()

def dense_block(x, units, drop):
    x = Dense(units, activation="swish", kernel_regularizer=l2(1e-4))(x)
    x = BatchNormalization()(x)
    return Dropout(drop)(x)

def build_finder(a_shape, feat_dim, b_shape, a_trainable, b_trainable):
    inp = Input((2,)); ai = ValueOfIndexLayer(0)(inp); bi = ValueOfIndexLayer(1)(inp)
    AL = IndexedValueLayer(a_shape, trainable=a_trainable, name="a_latents")
    BL = IndexedValueLayer(b_shape, trainable=b_trainable, name="b_latents")
    fin = Input((feat_dim,))
    x = concatenate([dense_block(AL(ai), 64, 0.2), dense_block(fin, 32, 0.2),
                     dense_block(BL(bi), 64, 0.2)])
    x = BatchNormalization()(x)
    x = dense_block(x, 128, 0.3); x = dense_block(x, 64, 0.2)
    m = Model([inp, fin], Dense(1, kernel_regularizer=l2(1e-4))(x))
    m.a_latents_layer = AL; m.b_latents_layer = BL
    return m

## 3. Cyclic training / 순환 학습

Two finders share the same latent tables and train alternately: one side is
trainable while the other is frozen. Cycle-level early stopping restores the
latent tables from the best cycle.

Finder 두 개가 같은 잠재특성 테이블을 공유하며 번갈아 학습합니다. 한 번에
한쪽만 열어두고 다른 쪽은 얼립니다. 검증 손실이 좋아지지 않으면 사이클 단위로
멈추고 가장 좋았던 사이클의 잠재특성을 복원합니다.

In [ ]:
# ================== 순환 이중 잠재특성 발견 ==================
# Finder 두 개가 같은 잠재특성 테이블을 공유하며 번갈아 학습합니다.
# 한 번에 한쪽만 열어두고 다른 쪽은 얼립니다 — 이 교대가 CDLD의 핵심입니다.
LATENT = 16
CYCLES, SUB_EPOCH, PATIENCE, BATCH, LR = 12, 2, 3, 256, 1e-3

fx = np.stack([entity_a, entity_b], 1).astype(np.float32)
a_lat = rng.random((N_A, LATENT)).astype(np.float32)
b_lat = rng.random((N_B, LATENT)).astype(np.float32)

fa = build_finder(a_lat.shape, D_FEAT, b_lat.shape, True, False)   # A만 학습
fb = build_finder(a_lat.shape, D_FEAT, b_lat.shape, False, True)   # B만 학습
for f in (fa, fb):
    f.a_latents_layer.set_value(a_lat); f.b_latents_layer.set_value(b_lat)
    f.compile(loss="mse", optimizer=Adam(LR))

tin, vin = [fx[tr], features[tr]], [fx[va], features[va]]
best, best_A, best_B, wait = np.inf, None, None, 0
for c in range(CYCLES):
    fa.b_latents_layer.set_value(fb.b_latents_layer.get_value())
    fa.fit(tin, target[tr], epochs=SUB_EPOCH, batch_size=BATCH,
           validation_data=(vin, target[va]), verbose=0)
    fb.a_latents_layer.set_value(fa.a_latents_layer.get_value())
    h = fb.fit(tin, target[tr], epochs=SUB_EPOCH, batch_size=BATCH,
               validation_data=(vin, target[va]), verbose=0)
    vl = float(h.history["val_loss"][-1]); mark = ""
    if vl < best - 1e-5:
        best, wait = vl, 0
        best_A = fa.a_latents_layer.get_value()
        best_B = fb.b_latents_layer.get_value()
        mark = "  <- best"
    else:
        wait += 1
    print(f"cycle {c+1:2d}/{CYCLES} val MSE {vl:.4f}{mark}", flush=True)
    if wait >= PATIENCE:
        print(f"early stop at cycle {c+1} (best 복원)"); break
AL, BL = best_A, best_B
print(f"\n발견된 잠재특성: A {AL.shape}, B {BL.shape}")

## 4. What goes into the predictor / 무엇을 예측기에 넣을 것인가

In [ ]:
# ================== 여기만 바꾸면 됩니다 (2/2) — 무엇을 예측기에 넣을까 ==================
# 원 논문의 결론: **관측 특성이 있고 기록이 희소한 쪽은 관측 특성으로,
# 관측 특성이 없고 기록이 풍부한 쪽은 잠재특성으로** 넣는 비대칭 설계가 낫습니다.
# 아래 세 조건을 비교해 자기 데이터에서도 그런지 확인하십시오.
def build_predictor(feat_dim, lat_dim, use_a_lat, use_b_lat):
    ins, xs = [], []
    if feat_dim:
        i = Input((feat_dim,)); ins.append(i); xs.append(dense_block(i, 32, 0.2))
    if use_a_lat:
        i = Input((lat_dim,)); ins.append(i); xs.append(dense_block(i, 64, 0.2))
    if use_b_lat:
        i = Input((lat_dim,)); ins.append(i); xs.append(dense_block(i, 64, 0.2))
    x = concatenate(xs) if len(xs) > 1 else xs[0]
    x = BatchNormalization()(x)
    x = dense_block(x, 128, 0.3); x = dense_block(x, 64, 0.2)
    return Model(ins, Dense(1, kernel_regularizer=l2(1e-4))(x))

def inputs_for(idx, use_feat, use_a_lat, use_b_lat):
    out = []
    if use_feat: out.append(features[idx])
    if use_a_lat: out.append(AL[entity_a[idx]])
    if use_b_lat: out.append(BL[entity_b[idx]])
    return out

CONDS = {
    "features only":            (True,  False, False),
    "features + A latent":      (True,  True,  False),
    "features + B latent":      (True,  False, True),   # <- 비대칭 설계
    "features + both":          (True,  True,  True),
}
rows = []
for name, (uf, ua, ub) in CONDS.items():
    m = build_predictor(D_FEAT if uf else 0, LATENT, ua, ub)
    m.compile(loss="mse", optimizer=Adam(LR))
    m.fit(inputs_for(tr, uf, ua, ub), target[tr], epochs=60, batch_size=BATCH,
          validation_data=(inputs_for(va, uf, ua, ub), target[va]),
          callbacks=[tf.keras.callbacks.EarlyStopping(
              patience=10, restore_best_weights=True)], verbose=0)
    p = m.predict(inputs_for(te, uf, ua, ub), verbose=0).ravel()
    rows.append({"condition": name,
                 "RMSE": float(np.sqrt(mean_squared_error(target[te], p))),
                 "R2": float(r2_score(target[te], p))})
    print(f"{name:24s} RMSE {rows[-1]['RMSE']:.4f} | R2 {rows[-1]['R2']:.4f}")
    tf.keras.backend.clear_session()

res = pd.DataFrame(rows)
print()
print(res.round(4).to_string(index=False))

## 5. Checklist before moving to your data / 자기 데이터로 옮길 때

In [ ]:
# ================== 자기 데이터로 옮길 때의 점검표 ==================
print("""
1. 개체당 기록 수를 먼저 세십시오.
   잠재특성 발견의 성패는 개체당 상호작용 수가 결정합니다. 원 연구에서는
   개체당 수십~수백 건이면 발견됐고, 한두 건이면 발견되지 않았습니다.
   기록이 희소한 쪽에 잠재특성을 주면 오히려 성능이 나빠질 수 있습니다.

2. 어느 쪽에 관측 특성이 있는지 확인하십시오.
   관측 특성이 없는 쪽이 잠재특성의 후보입니다. 둘 다 있으면 굳이 쓸 이유가
   적고, 둘 다 없으면 상호작용 기록량이 충분한지가 유일한 기준입니다.

3. 학습에 없던 개체는 별도 처리가 필요합니다.
   잠재특성은 학습 분할에 등장한 개체에 대해서만 정의됩니다(transductive).
   평균 벡터 대치, 재학습, 소량 기록으로 한 행만 재추정 등을 정하십시오.

4. 단순 집계 대조군을 반드시 함께 두십시오.
   개체별 목표값 평균 한 컬럼을 추가하는 것만으로 이득의 상당 부분이
   재현될 수 있습니다. 잠재특성이 그것을 넘는지 확인해야 기여를 주장할 수
   있습니다. 개체 식별자만 입력으로 쓰는 조건에서는 그 평균이 폐형 최적해라
   잠재특성이 이길 수 없다는 점도 함께 기억하십시오.

5. 분류 문제로 바꿀 때는 라벨 shape에 주의하십시오.
   Keras에서 1차원 라벨 (b,) 와 (b,1) 출력이 만나면 브로드캐스팅으로
   (b,b) 손실이 만들어집니다. 라벨을 항상 reshape(-1, 1) 하십시오.
""")

DOMAINS = pd.DataFrame([
    {"도메인": "재원일수 예측", "기록 희소 개체 A": "환자", "기록 풍부 개체 B": "의사",
     "target": "재원일수"},
    {"도메인": "교육 평가", "기록 희소 개체 A": "문항", "기록 풍부 개체 B": "학생",
     "target": "정오답"},
    {"도메인": "추천", "기록 희소 개체 A": "상품", "기록 풍부 개체 B": "고객",
     "target": "평점·구매"},
    {"도메인": "업무 관리", "기록 희소 개체 A": "과업·팀", "기록 풍부 개체 B": "직원",
     "target": "완료 시간·품질"},
])
print(DOMAINS.to_string(index=False))